<a href="https://colab.research.google.com/github/trialdyk/marchine-learning/blob/main/JS03/JS03_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Lab - Wisconsin Breast Cancer
## JS03 - Ekstraksi Fitur

**Dataset:** `wbc.csv` (569 data, target `diagnosis` M/B)

**Tugas:**
1. Pisahkan variabel yang dapat & tidak dapat digunakan
2. Encoding kolom `diagnosis`
3. Standarisasi semua kolom numerik
4. Seleksi fitur dengan SelectKBest
5. Pengujian dengan Logistic Regression
6. Membangun model pipeline
7. Analisis: jumlah fitur terbaik & fitur tersebut

## Langkah 0 - Library & Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print('Library loaded OK')

Library loaded OK


In [2]:
df = pd.read_csv('wbc.csv')
print('Shape:', df.shape)
df.head()

Shape: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## 1. Pisahkan Variabel yang Dapat & Tidak Dapat Digunakan

`id` (identitas unik) dan kolom terakhir yang seluruhnya kosong tidak dapat digunakan.

In [3]:
# Cek kolom yang tidak berguna
print('Kolom terakhir non-null:', df.iloc[:, -1].notna().sum())
print('Kolom id unik:', df['id'].nunique())

# Hapus kolom tanpa nama (kolom terakhir) + id
df = df.drop(columns=['id', df.columns[-1]])
print('Setelah drop, shape:', df.shape)

Kolom terakhir non-null: 0
Kolom id unik: 569
Setelah drop, shape: (569, 31)


## 2. Encoding kolom "diagnosis"

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

print('Mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
df['diagnosis'].value_counts()

Mapping: {'B': np.int64(0), 'M': np.int64(1)}


,count
diagnosis,
0,357
1,212


## 3 - 6. Pipeline: Standarisasi + Seleksi Fitur + Logistic Regression

In [5]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (455, 30) Test: (114, 30)


In [6]:
def build_pipe(k):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('sel', SelectKBest(score_func=f_classif, k=k)),
        ('clf', LogisticRegression(max_iter=1000))
    ])

# Uji dengan k = jumlah seluruh fitur (tanpa seleksi) sebagai baseline
pipe = build_pipe(k=X.shape[1])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print(f"=== Baseline: semua 30 fitur ===")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

=== Baseline: semua 30 fitur ===
Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## Eksperimen: Berapa Fitur Terbaik?

In [7]:
results = []
for k in range(1, X.shape[1] + 1):
    p = build_pipe(k)
    p.fit(X_train, y_train)
    acc = accuracy_score(y_test, p.predict(X_test))
    results.append((k, acc))

for k, acc in results:
    print(f'k={k:2d}  accurary={acc:.4f}')

k= 1  accurary=0.9298
k= 2  accurary=0.9561
k= 3  accurary=0.9561
k= 4  accurary=0.9561
k= 5  accurary=0.9649
k= 6  accurary=0.9649
k= 7  accurary=0.9649
k= 8  accurary=0.9649
k= 9  accurary=0.9737
k=10  accurary=0.9561
k=11  accurary=0.9737
k=12  accurary=0.9737
k=13  accurary=0.9737
k=14  accurary=0.9825
k=15  accurary=0.9737
k=16  accurary=0.9737
k=17  accurary=0.9737
k=18  accurary=0.9825
k=19  accurary=0.9825
k=20  accurary=0.9825
k=21  accurary=0.9737
k=22  accurary=0.9737
k=23  accurary=0.9737
k=24  accurary=0.9737
k=25  accurary=0.9737
k=26  accurary=0.9737
k=27  accurary=0.9737
k=28  accurary=0.9649
k=29  accurary=0.9649
k=30  accurary=0.9649


In [8]:
best = max(results, key=lambda t: t[1])
print('k terbaik:', best[0], 'akurasi:', best[1])

# Ambil daftar fitur k terbaik
pipe_best = build_pipe(best[0])
pipe_best.fit(X_train, y_train)
mask = pipe_best.named_steps['sel'].get_support()
print('\nFitur terpilih:')
print(X.columns[mask].tolist())

k terbaik: 14 akurasi: 0.9824561403508771

Fitur terpilih:
['radius_mean', 'perimeter_mean', 'area_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'radius_se', 'perimeter_se', 'radius_worst', 'perimeter_worst', 'area_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst']


## 7. Kesimpulan

Lengkapi berdasarkan sel hasil: fitur terbaik, akurasinya, dan kesimpulannya.